In [28]:
import numpy as np
import torch
import torch.nn as nn


In [29]:
torch.manual_seed(0)
np.random.seed(0)


In [30]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_c, out_c, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_c * out_c)
        self.weights = nn.Parameter(
            scale * torch.randn(in_c, out_c, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, _, N = x.shape
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(B, self.weights.shape[1], N // 2 + 1,
                             dtype=torch.cfloat, device=x.device)
        k = min(self.modes, N // 2 + 1)
        out_ft[:, :, :k] = torch.einsum("bix,iox->box",
                                        x_ft[:, :, :k], self.weights[:, :, :k])
        return torch.fft.irfft(out_ft, n=N)


class FNO1d(nn.Module):
    def __init__(self, modes=64, width=64, n_layers=4):
        super().__init__()
        self.lift = nn.Linear(2, width)
        self.spec = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(n_layers)])
        self.w    = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, x):
        x = self.lift(x).permute(0, 2, 1)
        for s, w in zip(self.spec, self.w):
            x = nn.functional.gelu(s(x) + w(x))
        x = x.permute(0, 2, 1)
        return self.proj(x).squeeze(-1)


def make_input(u):
    B, N = u.shape
    x = torch.linspace(0, 1, N + 1, device=u.device)[:-1]
    x = x.expand(B, N)
    return torch.stack([u, x], dim=-1)


In [31]:
def rel_l2_per_sample(pred, target):
    num = np.linalg.norm((pred - target).reshape(len(pred), -1), axis=1)
    den = np.linalg.norm(target.reshape(len(target), -1), axis=1) + 1e-8
    return num / den


In [32]:
device = 'cpu'
model_path = 'FNO_submission/fno_modes64_all48_seed0.pt'

ckpt = torch.load(model_path, map_location=device, weights_only=False)
model = FNO1d(**ckpt['config']).to(device)
model.load_state_dict(ckpt['state'])
model.eval()
print(f"Loaded {model_path}\n")


Loaded FNO_submission/fno_modes64_all48_seed0.pt



In [33]:
# apply the same trained model to all three resolutions, no retraining
results = {}
with torch.no_grad():
    for res in [128, 256, 512]:
        data = np.load(f'data/eval_{res}.npz')
        X = data['field'].astype(np.float32)
        Y = data['target'].astype(np.float32)

        pred = model(make_input(torch.tensor(X, device=device))).cpu().numpy()

        fno_err = rel_l2_per_sample(pred, Y)
        id_err = rel_l2_per_sample(X, Y)

        results[res] = {'fno': fno_err, 'identity': id_err}

        print(f"=== Resolution {res} ===")
        print(f"  FNO mean rel-L2   : {fno_err.mean():.6f}")
        print(f"  FNO median        : {np.median(fno_err):.6f}")
        print(f"  Identity baseline : {id_err.mean():.6f}")
        print(f"  Improvement factor: {id_err.mean() / fno_err.mean():.2f}x")
        print()


=== Resolution 128 ===
  FNO mean rel-L2   : 0.004045
  FNO median        : 0.000821
  Identity baseline : 0.032534
  Improvement factor: 8.04x

=== Resolution 256 ===
  FNO mean rel-L2   : 0.004271
  FNO median        : 0.001231
  Identity baseline : 0.032549
  Improvement factor: 7.62x

=== Resolution 512 ===
  FNO mean rel-L2   : 0.004332
  FNO median        : 0.001449
  Identity baseline : 0.032552
  Improvement factor: 7.51x



In [34]:
e128 = results[128]['fno'].mean()
e256 = results[256]['fno'].mean()
e512 = results[512]['fno'].mean()


In [35]:
e128 = results[128]['fno'].mean()
e256 = results[256]['fno'].mean()
e512 = results[512]['fno'].mean()


In [36]:
# per-example errors for the buckets 
np.savez('fno_errors.npz',
         err_128=results[128]['fno'],
         err_256=results[256]['fno'],
         err_512=results[512]['fno'],
         id_128=results[128]['identity'],
         id_256=results[256]['identity'],
         id_512=results[512]['identity'])
print("\nSaved per-example errors to fno_errors.npz")



Saved per-example errors to fno_errors.npz
